# Introduction 

This notebook is used to perform quick data analysis of the all experiments that are used for download. The goal of this notebook is to provide a quick answer on how many files are available in total and how many of them can be used for the deep learning 



In [3]:
import pandas as pd

# Sheffield patients


In [5]:
sheffield_df = pd.read_csv(r"PATH/TO/CSV")

In [6]:
sheffield_df.head()

,project_id,subject_label,experiment_label,scanner_type,series_descr_match,series_description,scan_id,n_frames
0,BEAt-DKD-WP4-Sheffield,IBE7128002,153847_20210913,Philips Healthcare Ingenia,True,DCE_kidneys_cor-oblique_fb,2801.0,0.0
1,BEAt-DKD-WP4-Sheffield,IBE7128002,153847_20210913,Philips Healthcare Ingenia,True,DCE_kidneys_cor-oblique_fb,3001.0,0.0
2,BEAt-DKD-WP4-Sheffield,IBE7128002,153847_20210913,Philips Healthcare Ingenia,True,DCE_kidneys_cor-oblique_fb,2801001.0,2.0
3,BEAt-DKD-WP4-Sheffield,IBE7128002,153847_20210913,Philips Healthcare Ingenia,True,DCE_kidneys_cor-oblique_fb,3001001.0,8.0
4,BEAt-DKD-WP4-Sheffield,IBE7128002,153847_20210913,Philips Healthcare Ingenia,True,DCE_kidneys_cor-oblique_fb,3001002.0,8.0


In [7]:
sheffield_df.tail()

,project_id,subject_label,experiment_label,scanner_type,series_descr_match,series_description,scan_id,n_frames
3586,BEAt-DKD-WP4-Sheffield,IBE7128065,154971_20220629,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,21.0,7168.0
3587,BEAt-DKD-WP4-Sheffield,IBE7128148,156741_20230804,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,23.0,7168.0
3588,BEAt-DKD-WP4-Sheffield,IBE7128155,156839_20230821,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,24.0,7168.0
3589,BEAt-DKD-WP4-Sheffield,IBE7128054,154803_20220601,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,22.0,7168.0
3590,BEAt-DKD-WP4-Sheffield,IBE7128141,156667_20230717,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,23.0,7168.0


In [8]:
sheffield_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3591 entries, 0 to 3590
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   project_id          3591 non-null   object 
 1   subject_label       3591 non-null   object 
 2   experiment_label    3591 non-null   object 
 3   scanner_type        3589 non-null   object 
 4   series_descr_match  3591 non-null   bool   
 5   series_description  3563 non-null   object 
 6   scan_id             3563 non-null   float64
 7   n_frames            3563 non-null   float64
dtypes: bool(1), float64(2), object(5)
memory usage: 200.0+ KB


The dataset contains 3591 entries. This does not mean that there are 1591 unique files. It only indicates that there is an error in the file storing and acqusition. Look at the first 5 entries of the `sheffield_df.head()` and compare them to the last 5 entried from the `sheffield_df.tail()`. It can be noriced that some entries have the same `subject_label` and `experiment_label` but they have different `scan_id` values. In other words, for some experiments the series is stored in bunch of small scans with small amount of frames rather than one huge series flder. We will address this later in the notebook

In [9]:
# calculating how many unique experiments are in the dataset (should be 140)
n_unique_exp_sheff = len(pd.unique(sheffield_df["experiment_label"]))
print(f"number of uniue recorded experiments before removing outliers: {n_unique_exp_sheff}")

number of uniue recorded experiments before removing outliers: 141


There are 140 experiments in total. You may be wondering from where this 1 extra input came from?

It comes from the IBE7128047 experiment. This experiment is the only one that has 2 experiment_labels: 154354_20220420_ORIG (has no relevant data) and 154354_20220420 (has the relevant data). From manual inspection of the whole Sheffield dataset, it can be confirmed that this is the only outlier. Hence it can be removed from the analysis of the Sheffield patients

In [ ]:
sheffield_df = sheffield_df.drop(sheffield_df[sheffield_df["experiment_label"] == ""].index)
n_unique_exp_sheff = len(pd.unique(sheffield_df["experiment_label"]))
print(f"number of uniue recorded experiments after removing the outlier: {n_unique_exp_sheff}")

number of uniue recorded experiments after removing the outlier: 140


Let's see how many unique experiments got matched with the series description (either `DCE_kidneys_cor-oblique_fb`, or `3D_DISCO_Dyn_kidneys_cor-oblique_fb`). These are the results that can be used for training and validating the deep learning algorithm

In [11]:
matched_df_sheff = sheffield_df[sheffield_df["series_descr_match"]==True] # dataframe containing only the results that have matched series
n_unique_matched_sheff = len(matched_df_sheff.drop_duplicates(subset="experiment_label"))

print(f"Number of unique experiments that matched the series description: {n_unique_matched_sheff}")

Number of unique experiments that matched the series description: 113


Aha! 113 is still a good number of examples. But are all of them useful? Well, not really. 

There are still 2 possible reasons why even the values that got matched with the series description still might need to be discarded:
1. **Single series explodes into bunch of smaller folders** each containing 8 frames instead of one single folder for the whole series (e.g. IBE7128002). *In this case, the scans can potentially be restored with additional processing*
2. **Series contains less than expected amount of frames**. For example, instead of 7168 expected frames there are 4032 or even aroud 500 frames. *In this case, the scans can potentially be used if the number of frames is not too short*




Addresing the `1` point. Good news! All "exploded" experiments can be tracked down to a single scanner `"Philips Healthcare Ingenia"`. Hence, the error is not random. This allows us to group them together in a single dataframe. Important to note, some scans like `IBE7128003` did not get any match but still are taken with this scanner. Since we are interested in exploring inconsistencies only among matched experiments, the analysis will be run on the `matched_df_sheff` DataFrame

In [13]:
ingenia_df_sheff = matched_df_sheff[matched_df_sheff["scanner_type"] == "Philips Healthcare Ingenia"]  # All ingenia scanner series from matched dataset

ingenia_unique_subj = ingenia_df_sheff.drop_duplicates(subset="subject_label") # get only unique values
ingenia_subj_lbls = ingenia_unique_subj["subject_label"] # get only subject labels for scans taken with ingenia

n_unique_ingenia_sheff = len(ingenia_unique_subj)


print(f"Number of matched files taken with the Philips Healthcare Ingenia scanner: {n_unique_ingenia_sheff}"
      f"\n list of all subject labels taken with this scanner"
      f"\n {ingenia_subj_lbls} ")


Number of matched files taken with the Philips Healthcare Ingenia scanner: 13
 list of all subject labels taken with this scanner
 0       IBE7128002
270     IBE7128005
536     IBE7128006
802     IBE7128007
1068    IBE7128008
1334    IBE7128009
1600    IBE7128010
1866    IBE7128011
2134    IBE7128012
2400    IBE7128014
2666    IBE7128015
2932    IBE7128016
3198    IBE7128017
Name: subject_label, dtype: object 


Now we have the list of files that we should exclude to get rid of the "exploded" series experiments taken with Philips Healthcare Ingenia. After excluding them, we can proceed with addressing 2nd point 

In [14]:
no_ingenia = matched_df_sheff.copy() # copy dataset to avoid accidentally deleting all the data
no_ingenia = no_ingenia.drop(ingenia_df_sheff.index) # drop all rows that are taken with ingenia scanner

print(f"length of the dataset when all ingenia scans are removed: {len(no_ingenia)}")

length of the dataset when all ingenia scans are removed: 101


How we can have 113 matched rows, 13 matched rows scanned with Ingenia, and then get 101 rows when all scans removed? This is a hint that there might be duplicates. For example, experiment that was not taken with ingeria still can have two scan_ids that match the description

In [15]:
dupes = no_ingenia[no_ingenia.duplicated("experiment_label", keep=False)]
dupes

,project_id,subject_label,experiment_label,scanner_type,series_descr_match,series_description,scan_id,n_frames
3554,BEAt-DKD-WP4-Sheffield,IBE7128129,156359_20230531,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,22.0,84.0
3555,BEAt-DKD-WP4-Sheffield,IBE7128129,156359_20230531,GE MEDICAL SYSTEMS SIGNA PET/MR,True,3D_DISCO_Dyn_kidneys_cor-oblique_fb,23.0,7168.0


Aha! It can be seen that IBE7128129 is a duplicate entry and that it has two series that have the same label but have 2 different number of frames. This brings us to the second point where not all series can be used if they have short number of frames. Here the series with `scan_id = 22` has only 84 frames, and it should be discarded from the dataset

In [16]:
# drop IBE7128129 with scan_id = 22.0
no_ingenia_no_dupes = no_ingenia[
    ~((no_ingenia["subject_label"] == "IBE7128129") 
      & (no_ingenia["scan_id"] == 22.0 ))]

print(f"Number of unique experiments without the ingenia scanner: {len(no_ingenia_no_dupes)}")

Number of unique experiments without the ingenia scanner: 100


Now we can proceed with finding the number of unique possible experiments

In [17]:
no_ingenia_no_dupes.nunique()

project_id              1
subject_label         100
experiment_label      100
scanner_type            1
series_descr_match      1
series_description      1
scan_id                 8
n_frames                6
dtype: int64

It can be seen that there are 6 different types of n_frames values. let's investigate

In [18]:
no_ingenia_no_dupes["n_frames"].value_counts().sort_index()

n_frames
588.0      1
2240.0     1
3556.0     1
4032.0     2
7168.0    94
7196.0     1
Name: count, dtype: int64

from this, we can say that most of the values have 7168 frames (94 of them)

In [ ]:
# Getting all frames that do not have length of 7168
df_bad_frames = no_ingenia_no_dupes[no_ingenia_no_dupes["n_frames"] != 7168.0]

# only keep subject_label, experiment_label, scan_id, n_frames
df_bad_frames_short = df_bad_frames[["subject_label", "experiment_label", "scan_id", "n_frames"]]
df_bad_frames_short.to_csv(r"PATH/TO/CSV", index=False)

df_bad_frames_short


,subject_label,experiment_label,scan_id,n_frames
3465,IBE7128020,154257_20211215,21.0,4032.0
3478,IBE7128035,154332_20220315,20.0,4032.0
3498,IBE7128066,154800_20220630,21.0,2240.0
3547,IBE7128116,155839_20230117,23.0,7196.0
3565,IBE7128144,156697_20230725,18.0,3556.0
3569,IBE7128146,156739_20230801,23.0,588.0


Now let's create separate data frame for the unmatched data



In [30]:
unmatched_df_sheff = sheffield_df[sheffield_df["series_descr_match"]==False]
print(len(unmatched_df_sheff))

unmatched_df_sheff.head()


27


,project_id,subject_label,experiment_label,scanner_type,series_descr_match,series_description,scan_id,n_frames
268,BEAt-DKD-WP4-Sheffield,IBE7128003,153848_20210914,Philips Healthcare Ingenia,False,NaN,NaN,NaN
269,BEAt-DKD-WP4-Sheffield,IBE7128004,153851_20210927,Philips Healthcare Ingenia,False,NaN,NaN,NaN
3467,BEAt-DKD-WP4-Sheffield,IBE7128025,154254_20220126,GE MEDICAL SYSTEMS SIGNA PET/MR,False,NaN,NaN,NaN
3479,BEAt-DKD-WP4-Sheffield,IBE7128043,154662_20220406,GE MEDICAL SYSTEMS SIGNA PET/MR,False,NaN,NaN,NaN
3485,BEAt-DKD-WP4-Sheffield,IBE7128048,154665_20220517,GE MEDICAL SYSTEMS SIGNA PET/MR,False,NaN,NaN,NaN


In [31]:
unmatched_df_sheff.to_csv(r"PATH/TO/CSV", index=False)